<img src="../Images/DSC_Logo.png" style="width: 400px;">

# Basics of Quantitative Data Analysis in Python
# - Structural Data Preprocessing

After loading and organizing data, the next step in quantitative research is often preprocessing. This means preparing the dataset so that it is consistent and ready for later analysis.

In this notebook, we focus on **structural data preprocessing** using the Iris dataset, a well-known example of tabular quantitative data. Because the original Iris dataset is already very clean, we use a deliberately messier version to explore common preprocessing steps in a more realistic way.

The focus is on:
- inspecting the dataset
- identifying data quality issues
- cleaning and standardizing values
- converting variables into appropriate formats
- checking the result after each step

While the main focus is on tabular data with `pandas`, the notebook also briefly previews how similar preprocessing ideas can be applied to arrays in Sect. 2.

In [1]:
# Install all required libraries for this notebook:

!pip install -q -r ../requirements_B.txt   

print("The libraries have been installed.")

The libraries have been installed.


## 1. Preprocessing Tabular Data


We first import `pandas` and read the messy Iris CSV file. This gives us a dataframe to inspect and preprocess:

In [2]:
import pandas as pd    # Import library (must be installed)

Iris = pd.read_csv("../Data/DataB/Iris_Messy.csv")

Iris.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,Petal Width (cm),Species,MeasurementDate
0,74,6.1,2.8,4.7,1.2,iris-versicolor,05-06-2024
1,19,5.7,3.8,1.7,0.3,Iris-setosa,2024-05-20
2,119,7.7,2.6,6.9,2.3,Iris-virginica,27/08/2024
3,79,6.0,2.9,4.5,1.5,Iris-versicolor,2024-06-10
4,77,6.8,2.8,4.8,1.4,Iris-versicolor,2024-06-08


<img src="../Images/iris.png" style="width: 780px;">

<small>*Image modified from Steve Dorand, Pixabay*</small>

## 1.1 Inspect the Dataset

Before changing anything, we **inspect** the dataset systematically. This helps us understand its structure, data types, and possible problems.

Compared to the original dataset, the messy dataset has 4 more rows:

In [3]:
print(Iris.shape)

(154, 7)


Compared to the original dataset, the messy dataset has an additional column (`MeasurementDate`) and the column `Petal Width (cm)` follows not the same naming convention as the other columns:

In [4]:
print(Iris.columns)

Index(['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm',
       'Petal Width (cm)', 'Species', 'MeasurementDate'],
      dtype='object')


`.dtypes` shows the data types of all columns. It is strange that the column `PetalLengthCm` has type `object`, since it should contain only numerical values.

In [5]:
print(Iris.dtypes)

Id                    int64
SepalLengthCm       float64
SepalWidthCm        float64
PetalLengthCm        object
Petal Width (cm)    float64
Species              object
MeasurementDate      object
dtype: object


`.info()` gives a compact overview of the dataframe. Here, for example, we can quickly see that `SepalWidthCm` and `PetalLengthCm` have missing values because they have only 152 non-null entries instead of 154.

In [6]:
Iris.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 154 entries, 0 to 153
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Id                154 non-null    int64  
 1   SepalLengthCm     154 non-null    float64
 2   SepalWidthCm      152 non-null    float64
 3   PetalLengthCm     152 non-null    object 
 4   Petal Width (cm)  154 non-null    float64
 5   Species           154 non-null    object 
 6   MeasurementDate   154 non-null    object 
dtypes: float64(3), int64(1), object(3)
memory usage: 8.6+ KB


`.describe()` provides summary statistics, which can help reveal suspicious patterns. Here, for example, we can see missing values in `SepalWidthCm`, while `PetalLengthCm` shows an unexpected `object` type and therefore does not behave like a normal numeric column in the summary.

In addition, we see:
- There is a negative minimum in `Petal Width (cm)`, and an implausibly large maximum of 9999 in `SepalLengthCm`.
- There are 9 unique species, however, the original dataset has only 3 unique species.

In [7]:
Iris.describe(include="all")

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,Petal Width (cm),Species,MeasurementDate
count,154.000000,154.000000,152.000000,152,154.000000,154,154
unique,NaN,NaN,NaN,44,NaN,9,130
top,NaN,NaN,NaN,1.5,NaN,Iris-setosa,2024-06-18
freq,NaN,NaN,NaN,14,NaN,49,3
mean,75.538961,70.736364,3.055263,NaN,1.193506,NaN,NaN
std,43.133799,805.271737,0.440016,NaN,0.763007,NaN,NaN
min,1.000000,4.300000,2.000000,NaN,-0.200000,NaN,NaN
25%,39.250000,5.100000,2.800000,NaN,0.300000,NaN,NaN
50%,75.500000,5.800000,3.000000,NaN,1.300000,NaN,NaN
75%,112.750000,6.400000,3.300000,NaN,1.800000,NaN,NaN


Let's inspect the unique labels in the `Species` column! It contains inconsistent labels such as different capitalization, spacing, and naming styles.

In [8]:
Iris["Species"].unique()

array(['iris-versicolor ', 'Iris-setosa', 'Iris-virginica',
       'Iris-versicolor', 'versicolor', 'setosa', 'iris virginica',
       'Iris Setosa', 'virginica'], dtype=object)

## 1.2 Rename Columns and Labels in Columns

We want to standardize the labels in `Species` so that the same category is not represented in different ways. There are multiple ways to do this in `pandas`, for example by using **string methods** to harmonize formatting, and by using a replacement **dictionary** to map different label variants to one consistent form.

Standardize general formatting:

In [9]:
# .str.strip() removes leading or trailing spaces
Iris["Species"] = Iris["Species"].str.strip()   

Iris["Species"].unique()

array(['iris-versicolor', 'Iris-setosa', 'Iris-virginica',
       'Iris-versicolor', 'versicolor', 'setosa', 'iris virginica',
       'Iris Setosa', 'virginica'], dtype=object)

Map all variants to the final desired labels:

In [10]:
# Create dictionary:
species_mapping = {
    "iris-versicolor": "Iris-versicolor",
    "versicolor": "Iris-versicolor",
    "setosa": "Iris-setosa",
    "Iris Setosa": "Iris-setosa",
    "iris virginica": "Iris-virginica",
    "virginica": "Iris-virginica"
}

# Use dictionary in .replace() function:
Iris["Species"] = Iris["Species"].replace(species_mapping)

Iris["Species"].unique()

array(['Iris-versicolor', 'Iris-setosa', 'Iris-virginica'], dtype=object)

One column name does not fit the naming style of the others. We use `.rename()` to rename it so that all column names are consistent. Since we only want to change a single column name, we can pass the old and new name directly as a small dictionary inside `.rename()` instead of defining a separate dictionary first.

In [11]:
# Replace column name
Iris = Iris.rename(columns={"Petal Width (cm)": "PetalWidthCm"})

print(Iris.columns)

Index(['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm',
       'Species', 'MeasurementDate'],
      dtype='object')


## 1.3 Check Missing Values

Next, we examine missing values in the dataset. Missing values are common in real-world datasets and should first be identified and understood. Depending on the context, they may later be left as missing, removed, or replaced — but the first step is always to inspect them systematically.

In `pandas`, `.isna()` and `.isnull()` can be used to detect missing entries. They do the same thing and return `True` where a value is missing and `False` otherwise. Combined with methods such as `.sum()` or row filtering, they help us count and inspect where missing values occur. 

In this messy version, missing values were introduced only in one column, but we first check the dataset systematically:

- Show the full dataframe of True/False values:

In [12]:
Iris.isna()     # alternative: Iris.isnull()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species,MeasurementDate
0,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...
149,False,False,False,False,False,False,False
150,False,False,False,False,False,False,False
151,False,False,False,False,False,False,False
152,False,False,True,False,False,False,False


- Count how many missing values each column contains:

In [13]:
Iris.isna().sum()

Id                 0
SepalLengthCm      0
SepalWidthCm       2
PetalLengthCm      2
PetalWidthCm       0
Species            0
MeasurementDate    0
dtype: int64

- Show only rows that contain at least one missing value anywhere in the row:

In [14]:
Iris[Iris.isna().any(axis=1)]

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species,MeasurementDate
26,56,5.7,NaN,4.5,1.3,Iris-versicolor,2024/05/18
34,84,6.0,2.7,NaN,1.6,Iris-versicolor,2024-06-15
61,137,6.3,3.4,NaN,2.4,Iris-virginica,14/09/2024
152,56,5.7,NaN,4.5,1.3,Iris-versicolor,2024/05/18


## 1.4 Clean and Convert Values

The column `PetalLengthCm` should be numeric, but some values were changed into text formats. A general workflow is to first identify problematic entries, then clean them, and finally convert the column to a numeric type.

- First, we identify values that cannot be converted directly. By trying to convert the column with `pd.to_numeric(..., errors="coerce")`, we can identify entries that are not directly numeric. Here, the entry `5.1 cm` cannot be converted because it contains text, and the missing values also appear as `NaN`.

In [15]:
petal_length_numeric = pd.to_numeric(Iris["PetalLengthCm"], errors="coerce")

Iris[petal_length_numeric.isna()][["Id", "PetalLengthCm"]]

,Id,PetalLengthCm
32,111,5.1 cm
34,84,NaN
61,137,NaN


- We now clean the non-numeric formatting and convert the column again. The entry `5.1 cm` becomes a valid number after removing the unit, while the remaining missing values stay as `NaN`. We use string methods to remove the string `" cm"` from the value:

In [16]:
Iris["PetalLengthCm"] = Iris["PetalLengthCm"].astype(str).str.replace(" cm", "", regex=False)

- Convert the column to numeric:

In [17]:
Iris["PetalLengthCm"] = pd.to_numeric(Iris["PetalLengthCm"], errors="coerce")

# Check the result
print(Iris["PetalLengthCm"].dtype)
Iris.loc[Iris["PetalLengthCm"].isna(), ["Id", "PetalLengthCm"]]

float64


,Id,PetalLengthCm
34,84,NaN
61,137,NaN


## 1.5 Handle / Convert Date Formats

The `MeasurementDate` column contains dates in several different formats: `2024-05-20`, `27/08/2024`, `2024/05/31`, and `05-06-2024`. Some dates start with the year, others with the day, and both slashes and hyphens are used as separators.

In [18]:
print(Iris["MeasurementDate"])

0      05-06-2024
1      2024-05-20
2      27/08/2024
3      2024-06-10
4      2024-06-08
          ...    
149    2024-05-24
150    2024-05-26
151    2024/05/14
152    2024/05/18
153    2024-06-18
Name: MeasurementDate, Length: 154, dtype: object


To handle this inconsistency, we convert the column to a proper datetime type with `to_datetime()`.
- The argument `dayfirst=True` mainly affects dates that start with day and month, especially ambiguous ones such as `05-06-2024`. 
- Dates that already clearly start with the year, such as `2024/05/31`, are still interpreted correctly, because `dayfirst=True` only matters when the parser has to decide whether the first number is the day or the month.

Because date formats can be ambiguous, it is important to inspect the converted result afterwards rather than assuming that every value was interpreted correctly.

In [19]:
Iris["MeasurementDate"] = pd.to_datetime(
    Iris["MeasurementDate"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Check the result
print(Iris["MeasurementDate"].dtype)
print(Iris["MeasurementDate"])
Iris.loc[Iris["MeasurementDate"].isna(), ["Id", "MeasurementDate"]]     # show rows where MeasurementDate is missing (because it could not be converted to datetime)

datetime64[ns]
0     2024-06-05
1     2024-05-20
2     2024-08-27
3     2024-06-10
4     2024-06-08
         ...    
149   2024-05-24
150   2024-05-26
151   2024-05-14
152   2024-05-18
153   2024-06-18
Name: MeasurementDate, Length: 154, dtype: datetime64[ns]


,Id,MeasurementDate


## 1.6 Implausible Numeric Values

Another useful preprocessing step is to look for values that are technically numeric but still suspicious or invalid in the context of the dataset. In this messy Iris dataset, this includes a negative petal width in column `PetalWidthCm` and the placeholder value 9999 for missing data in the column `SepalLengthCm`.

- Show rows with negative petal width:

In [20]:
Iris.loc[Iris["PetalWidthCm"] < 0]

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species,MeasurementDate
12,24,5.1,3.3,1.7,-0.2,Iris-setosa,2024-09-03


- Show rows where `SepalLengthCm` has the placeholder value 9999:

In [21]:
Iris.loc[Iris["SepalLengthCm"] == 9999]

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species,MeasurementDate
70,95,9999.0,2.7,4.2,1.3,Iris-versicolor,2024-06-26


We replace these values with missing values:

In [22]:
Iris.loc[Iris["PetalWidthCm"] < 0, "PetalWidthCm"] = pd.NA
Iris.loc[Iris["SepalLengthCm"] == 9999, "SepalLengthCm"] = pd.NA

# Check the result
print(f"Rows with missing values in PetalWidthCm:")
print(Iris.loc[Iris["PetalWidthCm"].isna(), ["Id", "PetalWidthCm"]])
print(f"\nRows with missing values in SepalLengthCm:")
print(Iris.loc[Iris["SepalLengthCm"].isna(), ["Id", "SepalLengthCm"]])

Rows with missing values in PetalWidthCm:
    Id  PetalWidthCm
12  24           NaN

Rows with missing values in SepalLengthCm:
    Id  SepalLengthCm
70  95            NaN


## 1.7 Duplicate Rows

Another common preprocessing step is to check whether the same observation appears more than once. Duplicate rows can result from merging files, repeated entries, or import errors. 

Duplicate rows are not automatically wrong. In some datasets, repeated rows may represent valid repeated observations. Whether duplicates should be removed depends on the meaning of the data. In this messy Iris dataset, however, exact duplicate rows are unintended and should be removed.

- Count duplicate rows:

In [23]:
Iris.duplicated().sum()

4

- Show duplicate rows:

In [24]:
Iris[Iris.duplicated(keep=False)]   # to show all occurrences of duplicated rows, not only the later repeated ones, use keep=False

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species,MeasurementDate
26,56,5.7,NaN,4.5,1.3,Iris-versicolor,2024-05-18
69,45,5.1,3.8,1.9,0.4,Iris-setosa,2024-06-15
88,120,6.0,2.2,5.0,1.5,Iris-virginica,2024-08-28
122,87,6.7,3.1,4.7,1.5,Iris-versicolor,2024-06-18
145,45,5.1,3.8,1.9,0.4,Iris-setosa,2024-06-15
147,120,6.0,2.2,5.0,1.5,Iris-virginica,2024-08-28
152,56,5.7,NaN,4.5,1.3,Iris-versicolor,2024-05-18
153,87,6.7,3.1,4.7,1.5,Iris-versicolor,2024-06-18


Remove duplicate rows. Here, `.drop_duplicates()` keeps the first occurrence of each row and removes any exact duplicates that follow:

In [25]:
Iris = Iris.drop_duplicates()

# Check the result
Iris.duplicated().sum()

0

## 1.8 Sort Data

Because the rows in the messy dataset were shuffled, we now sort the dataframe by the `Id` column to restore the familiar order of the original Iris dataset.

In [26]:
Iris = Iris.sort_values("Id", ascending=True)

print(Iris)

      Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm  \
100    1            5.1           3.5            1.4           0.2   
43     2            4.9           3.0            1.4           0.2   
58     3            4.7           3.2            1.3           0.2   
106    4            4.6           3.1            1.5           0.2   
128    5            5.0           3.6            1.4           0.2   
..   ...            ...           ...            ...           ...   
31   146            6.7           3.0            5.2           2.3   
46   147            6.3           2.5            5.0           1.9   
41   148            6.5           3.0            5.2           2.0   
126  149            6.2           3.4            5.4           2.3   
118  150            5.9           3.0            5.1           1.8   

            Species MeasurementDate  
100     Iris-setosa      2024-05-02  
43      Iris-setosa      2024-05-03  
58      Iris-setosa      2024-05-04  
106    

If you want, you can also reset the row index afterwards:

In [27]:
Iris = Iris.sort_values("Id").reset_index(drop=True)

print(Iris)

      Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm  \
0      1            5.1           3.5            1.4           0.2   
1      2            4.9           3.0            1.4           0.2   
2      3            4.7           3.2            1.3           0.2   
3      4            4.6           3.1            1.5           0.2   
4      5            5.0           3.6            1.4           0.2   
..   ...            ...           ...            ...           ...   
145  146            6.7           3.0            5.2           2.3   
146  147            6.3           2.5            5.0           1.9   
147  148            6.5           3.0            5.2           2.0   
148  149            6.2           3.4            5.4           2.3   
149  150            5.9           3.0            5.1           1.8   

            Species MeasurementDate  
0       Iris-setosa      2024-05-02  
1       Iris-setosa      2024-05-03  
2       Iris-setosa      2024-05-04  
3      

---
### **Exercise: Preprocessing the World Happiness Report Dataset**

In this exercise, you will apply several preprocessing steps to a messy version of the [World Happiness Report 2024 dataset](https://www.worldhappiness.report/ed/2024/) (`World-happiness-report-2024_Messy.xlsx`).

The goal is to create a cleaned and standardized dataset, sort it by `Ladder score`, and save the result as a new file.

In the messy version:
- some labels in `Regional indicator` are inconsistent,
- some missing values are written as text instead of stored as true missing values,
- and values in `Ladder score` need to be checked and cleaned before sorting.

Your tasks step-by-step:
- importing the dataset
- inspecting its structure and identifying potential issues
- applying preprocessing steps to clean and standardize the data
- sorting the cleaned dataset by `Ladder score`
- saving the cleaned version of the dataset as CSV file (see B01)

In [28]:
# Solution (example):

# 1. Import the dataset
import pandas as pd

report = pd.read_excel("../Data/DataB/World-happiness-report-2024_Messy.xlsx")

print(report)

    Country name            Regional indicator Ladder score  upperwhisker  \
0         Israel  Middle East and North Africa        7.341         7.405   
1    Netherlands                western europe        7.319         7.383   
2         Norway                western europe        7.302         7.389   
3     Luxembourg                western europe        7.122         7.213   
4    Switzerland                western europe         7.06         7.147   
..           ...                           ...          ...           ...   
138  Afghanistan                    South Asia        1.721         1.775   
139      Finland                western europe        7.741         7.815   
140      Denmark                western europe       7..583         7.665   
141      Iceland                western europe        7.525         7.618   
142       Sweden                western europe        7.344         7.422   

     lowerwhisker Log GDP per capita Social support Healthy life expectancy

In [29]:
# 2. Inspect the dataset
report.info()

report.describe(include="all")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143 entries, 0 to 142
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Country name                  143 non-null    object 
 1   Regional indicator            143 non-null    object 
 2   Ladder score                  143 non-null    object 
 3   upperwhisker                  143 non-null    float64
 4   lowerwhisker                  143 non-null    float64
 5   Log GDP per capita            142 non-null    object 
 6   Social support                142 non-null    object 
 7   Healthy life expectancy       142 non-null    object 
 8   Freedom to make life choices  142 non-null    object 
 9   Generosity                    142 non-null    object 
 10  Perceptions of corruption     142 non-null    object 
 11  Dystopia + residual           142 non-null    object 
dtypes: float64(2), object(10)
memory usage: 13.5+ KB


,Country name,Regional indicator,Ladder score,upperwhisker,lowerwhisker,Log GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption,Dystopia + residual
count,143,143,143.000,143.000000,143.000000,142.000,142.000,142.000,142.000,142.000,142.000,142.000
unique,143,11,140.000,NaN,NaN,136.000,126.000,121.000,124.000,112.000,114.000,136.000
top,Israel,Sub-Saharan Africa,3.502,NaN,NaN,1.361,1.277,0.638,0.641,0.112,0.078,1.907
freq,1,35,2.000,NaN,NaN,2.000,3.000,3.000,2.000,4.000,3.000,3.000
mean,NaN,NaN,NaN,5.641175,5.413972,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,1.155008,1.187133,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,1.775000,1.667000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,4.845500,4.606000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,5.895000,5.674000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,6.507500,6.319000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# 3. Standardize labels in Regional indicator
print(report["Regional indicator"].unique())

report["Regional indicator"] = report["Regional indicator"].replace({
    "western europe": "Western Europe",
    "Central and East Europe": "Central and Eastern Europe"
})

print(report["Regional indicator"].unique())

['Middle East and North Africa' 'western europe' 'North America and ANZ'
 'Latin America and Caribbean' 'Central and Eastern Europe'
 'Central and East Europe' 'Southeast Asia' 'East Asia'
 'Commonwealth of Independent States' 'Sub-Saharan Africa' 'South Asia']
['Middle East and North Africa' 'Western Europe' 'North America and ANZ'
 'Latin America and Caribbean' 'Central and Eastern Europe'
 'Southeast Asia' 'East Asia' 'Commonwealth of Independent States'
 'Sub-Saharan Africa' 'South Asia']


In [31]:
# 4. Replace text placeholders for missing data with true missing values
report = report.replace({
    "no value": pd.NA,
    "missing value": pd.NA,
    "nan": pd.NA
})

report.isna().sum()

Country name                    0
Regional indicator              0
Ladder score                    0
upperwhisker                    0
lowerwhisker                    0
Log GDP per capita              3
Social support                  3
Healthy life expectancy         3
Freedom to make life choices    3
Generosity                      3
Perceptions of corruption       3
Dystopia + residual             3
dtype: int64

In [32]:
# 5. Clean and convert Ladder score

# Try converting Ladder score to numeric without changing it yet:
ladder_score_numeric = pd.to_numeric(report["Ladder score"], errors="coerce")

# Show rows where conversion fails:
report[ladder_score_numeric.isna()][["Country name", "Ladder score"]]

# Clean the formatting problem and convert again:
report["Ladder score"] = report["Ladder score"].astype(str).str.replace("..", ".", regex=False)
report["Ladder score"] = pd.to_numeric(report["Ladder score"], errors="coerce")

# Check the result:
print(report["Ladder score"].dtype)
report.loc[report["Ladder score"].isna(), ["Country name", "Ladder score"]]

float64


,Country name,Ladder score


In [33]:
# 6. Sort the dataset by Ladder score
report = report.sort_values("Ladder score", ascending=False).reset_index(drop=True)
print(report)

         Country name            Regional indicator  Ladder score  \
0             Finland                Western Europe         7.741   
1             Denmark                Western Europe         7.583   
2             Iceland                Western Europe         7.525   
3              Sweden                Western Europe         7.344   
4              Israel  Middle East and North Africa         7.341   
..                ...                           ...           ...   
138  Congo (Kinshasa)            Sub-Saharan Africa         3.295   
139      Sierra Leone            Sub-Saharan Africa         3.245   
140           Lesotho            Sub-Saharan Africa         3.186   
141           Lebanon  Middle East and North Africa         2.707   
142       Afghanistan                    South Asia         1.721   

     upperwhisker  lowerwhisker Log GDP per capita Social support  \
0           7.815         7.667              1.844          1.572   
1           7.665         7.500  

In [34]:
# 7. Save the cleaned dataset as CSV file
report.to_csv("../Data/DataB/World-happiness-report-2024_Cleaned.csv", index=False)

---

## 2. A Brief Look at Preprocessing Data Arrays

When preprocessing is applied to numerical arrays, the general idea is very similar to preprocessing tabular data: we inspect the data, identify problematic values or patterns, and then clean or standardize them. Typical steps may include detecting missing values or placeholder codes, identifying implausible values, and replacing or masking such entries. These ideas are briefly illustrated below. The main difference is that arrays are not organized in named columns like dataframes, so preprocessing is usually done through indexing, boolean masks, and `numpy` functions.

Let's create a small numerical array. Here, the value `-999` is used as a placeholder for missing data:

In [35]:
import numpy as np      # Import library (must be installed)

In [36]:
data = np.array([
    [1.2, 2.5, -999],
    [2.1, 3.3, 4.1],
    [1.9, -999, 2.7]
], dtype=float)

print(data)

[[   1.2    2.5 -999. ]
 [   2.1    3.3    4.1]
 [   1.9 -999.     2.7]]


To replace the placeholder values with `NaN` we make use of boolean indexing:

In [37]:
data[data == -999] = np.nan

print(data)

[[1.2 2.5 nan]
 [2.1 3.3 4.1]
 [1.9 nan 2.7]]


Check where values are missing by returning a boolean array showing which positions contain missing values:

In [38]:
np.isnan(data)

array([[False, False,  True],
       [False, False, False],
       [False,  True, False]])

Count how many missing values are present in the array:

In [39]:
np.isnan(data).sum()

2

Show (potentially implausible) values smaller than 0. In this example, there should no longer be any negative values after replacing -999.

In [40]:
data[data < 0]

array([], dtype=float64)

## Key Takeaways

In this notebook, you learned that:

- preprocessing means preparing data so that it is consistent and usable for later analysis,
- a good workflow is to inspect data first, then identify problems, clean them, and check the result,
- common preprocessing steps include renaming columns, harmonizing labels, detecting missing values, converting data types, handling dates, identifying implausible values, removing duplicates, and sorting data,
- many preprocessing ideas also apply to numerical arrays.